In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import librosa
import os
from IPython.display import Audio
import matplotlib.pyplot as plt
import soundfile as sf
import json

In [11]:
from models.tts.gpt_tts.gpt_tts import GPTTTS
from models.tts.gpt_tts.g2p_old_en import process, PHPONE2ID
from g2p_en import G2p
from models.codec.codec_latent.codec_latent import LatentCodecEncoder, LatentCodecDecoderWithTimbre
from models.codec.amphion_codec.codec import CodecEncoder, CodecDecoder
from utils.util import load_config

In [12]:
cfg = load_config("egs/tts/LaCoTTS/exp_config_base.json")
print(cfg)

{'model_type': 'GPTTTS', 'dataset': ['Your Dataset Name'], 'preprocess': {'hop_size': 200, 'sample_rate': 16000, 'processed_dir': '', 'valid_file': 'valid.json', 'train_file': 'train.json'}, 'model': {'latent_codec': {'encoder': {'d_mel': 128, 'd_model': 96, 'num_blocks': 4, 'out_channels': 256, 'use_tanh': False, 'pretrained_ckpt': 'ckpts/latent_codec/latent_codec_enc.bin'}, 'decoder': {'in_channels': 256, 'num_quantizers': 1, 'codebook_size': 8192, 'codebook_dim': 8, 'quantizer_type': 'fvq', 'use_l2_normlize': True, 'vocos_dim': 512, 'vocos_intermediate_dim': 4096, 'vocos_num_layers': 16, 'ln_before_vq': True, 'use_pe': False, 'pretrained_ckpt': 'ckpts/latent_codec/latent_codec_dec.bin'}}, 'wav_codec': {'encoder': {'d_model': 96, 'out_channels': 128, 'up_ratios': [2, 4, 5, 5], 'use_tanh': False, 'pretrained_ckpt': 'ckpts/wav_codec/wav_codec_enc.bin'}, 'decoder': {'in_channels': 128, 'upsample_initial_channel': 1536, 'num_quantizers': 8, 'codebook_size': 1024, 'codebook_dim': 128, 'qu

# Audio Tokenizer: Convert Speech to Token

In [13]:
wav_codec_enc = CodecEncoder(
    cfg=cfg.model.wav_codec.encoder
)

wav_codec_dec = CodecDecoder(
    cfg=cfg.model.wav_codec.decoder
)

latent_codec_enc = LatentCodecEncoder(
    cfg=cfg.model.latent_codec.encoder
)

latent_codec_dec = LatentCodecDecoderWithTimbre(
    cfg=cfg.model.latent_codec.decoder
)

wav_codec_enc.load_state_dict(torch.load("ckpts/wav_codec/wav_codec_enc.bin"))
wav_codec_dec.load_state_dict(torch.load("ckpts/wav_codec/wav_codec_dec.bin"))
latent_codec_enc.load_state_dict(torch.load("ckpts/latent_codec/latent_codec_enc.bin"))
latent_codec_dec.load_state_dict(torch.load("ckpts/latent_codec/latent_codec_dec.bin"))

wav_codec_enc.eval()
wav_codec_dec.eval()
latent_codec_enc.eval()
latent_codec_dec.eval()

wav_codec_enc.cuda()
wav_codec_dec.cuda()
latent_codec_enc.cuda()
latent_codec_dec.cuda()

# requires_grad false
wav_codec_enc.requires_grad_(False)
wav_codec_dec.requires_grad_(False)
latent_codec_enc.requires_grad_(False)
latent_codec_dec.requires_grad_(False)

LatentCodecDecoderWithTimbre(
  (quantizer): ResidualVQ(
    (quantizers): ModuleList(
      (0): FactorizedVectorQuantize(
        (in_project): Conv1d(256, 8, kernel_size=(1,), stride=(1,))
        (out_project): Conv1d(8, 256, kernel_size=(1,), stride=(1,))
        (codebook): Embedding(8192, 8)
      )
    )
  )
  (model): Sequential(
    (0): VocosBackbone(
      (embed): Conv1d(256, 512, kernel_size=(7,), stride=(1,), padding=(3,))
      (norm): LayerNorm((512,), eps=1e-06, elementwise_affine=True)
      (convnext): ModuleList(
        (0-15): 16 x ConvNeXtBlock(
          (dwconv): Conv1d(512, 512, kernel_size=(7,), stride=(1,), padding=(3,), groups=512)
          (norm): LayerNorm((512,), eps=1e-06, elementwise_affine=True)
          (pwconv1): Linear(in_features=512, out_features=4096, bias=True)
          (act): GELU(approximate='none')
          (pwconv2): Linear(in_features=4096, out_features=512, bias=True)
        )
      )
      (final_layer_norm): LayerNorm((512,), eps=

# Latent Codec Language Model TTS

In [14]:
gpt_tts = GPTTTS(cfg=cfg.model.gpt_tts)
gpt_tts.load_state_dict(torch.load("ckpts/gpt_tts/latent_codec_gpt_tts.bin", map_location="cpu"))

<All keys matched successfully>

In [15]:
gpt_tts.eval()
gpt_tts.cuda()
gpt_tts.requires_grad_(False)

GPTTTS(
  (model): LlamaForCausalLM(
    (model): LlamaModel(
      (embed_tokens): Embedding(8458, 1024, padding_idx=8448)
      (layers): ModuleList(
        (0-11): 12 x LlamaDecoderLayer(
          (self_attn): LlamaAttention(
            (q_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (o_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (rotary_emb): LlamaRotaryEmbedding()
          )
          (mlp): LlamaMLP(
            (gate_proj): Linear(in_features=1024, out_features=4096, bias=False)
            (up_proj): Linear(in_features=1024, out_features=4096, bias=False)
            (down_proj): Linear(in_features=4096, out_features=1024, bias=False)
            (act_fn): SiLU()
          )
          (input_layernorm): LlamaRMSNorm()
          (post_attention_layernorm): Llama

# Inference

In [16]:
# load reference speech
source_wav = librosa.load("/blob/v-zeqianju/dataset/tts/librispeech/test/ref_3s_new_diffprompt/Wave16k16bNormalized/00003.wav", sr=16000)[0]
# source_wav = source_wav[:len(source_wav) - 13000]
Audio(source_wav, rate=16000)

In [17]:
# text tokenize
source_text = "BUT WAS THAT ALL HER REWARD ONE OF THE LADIES"
target_text = "Where is a good place to eat around here?"
text = source_text + " " + target_text
g2p = G2p()
txt_struct, txt = process(text, g2p)
phone_seq = [p for w in txt_struct for p in w[1]]
phone_id = [PHPONE2ID[p] for p in phone_seq]
phone_id = torch.LongTensor(phone_id).unsqueeze(0).cuda()
phone_id.shape

torch.Size([1, 75])

In [18]:
# speech tokenize
prompt_wav = torch.FloatTensor(source_wav).unsqueeze(0).to("cuda")
# wav to latent
vq_emb = wav_codec_enc(prompt_wav.unsqueeze(1))
vq_emb = latent_codec_enc(vq_emb)
# latent to token
(
    _,
    vq_indices,
    _,
    _,
    _,
    speaker_embedding,
) = latent_codec_dec(vq_emb, vq=True, eval_vq=False, return_spk_embs=True)
prompt_id = vq_indices[0,:,:]
prompt_id.shape

torch.Size([1, 272])

In [19]:
gen_tokens = gpt_tts.sample_hf(
    phone_id,
    prompt_id,
    max_length=3600,
    temperature=0.9,
    top_k=8192,
    top_p=0.85,
    repeat_penalty=1.0,
    classifer_free_guidance=1.0,
)

In [20]:
# gen token to latent
vq_post_emb = latent_codec_dec.vq2emb(gen_tokens.unsqueeze(0))
recovered_latent = latent_codec_dec(
    vq_post_emb, vq=False, speaker_embedding=speaker_embedding
)

In [21]:
# reconvered latent to wav
recovered_audio = wav_codec_dec(recovered_latent, vq=False)

In [22]:
sf.write("examples/recon/4.wav", recovered_audio.squeeze().cpu().numpy(), 16000)
Audio("examples/recon/4.wav", rate=16000)

In [23]:
def get_noise_scale(wavform, noise, snr_min=5, snr_max=10):
    wavform_power = np.sum(wavform**2) / len(wavform)
    noise_power = np.sum(noise**2) / len(noise)
    snr = np.random.rand() * (snr_max - snr_min) + snr_min
    noise_scale = np.sqrt(wavform_power / (noise_power * 10 ** (snr / 10)))
    return noise_scale, snr


def calculate_snr(wavform, noise):
    wavform_power = np.sum(wavform**2) / len(wavform)
    noise_power = np.sum(noise**2) / len(noise)
    snr = 10 * math.log10(wavform_power / noise_power)
    return snr

def gen_env_speech(speech, env, snr_min, snr_max):
    noise_scale, snr = get_noise_scale(speech, env, snr_min, snr_max)
    return speech + env * noise_scale

In [24]:
audiocaps_wav_path = "/blob/v-yuancwang/TTADataset/processed_data/AudioCaps/wav"
audiocaps_info_path = "/blob/v-yuancwang/TTADataset/processed_data/AudioCaps/train.json"
with open(audiocaps_info_path, "r") as f:
    audiocaps_info = json.load(f)
for i, info in enumerate(audiocaps_info):
    if "traffic" in info["Caption"]:
        print(i, info) 

66 {'Dataset': 'AudioCaps', 'Uid': '-10fWp7Pqs4_30000_40000', 'Caption': 'Outside traffic passes by both close and distant'}
323 {'Dataset': 'AudioCaps', 'Uid': '-8aWuKAYgfM_170000_180000', 'Caption': 'A busy road with traffic going by and then a vehicle honks a horn'}
801 {'Dataset': 'AudioCaps', 'Uid': '-Lls6pBgI0k_10000_20000', 'Caption': 'Child speaking in a foreign language outside near traffic'}
808 {'Dataset': 'AudioCaps', 'Uid': '-LuXgQJ1owA_50000_60000', 'Caption': 'Motor vehicle, traffic, road noise'}
875 {'Dataset': 'AudioCaps', 'Uid': '-NO2z0vcBsA_200000_210000', 'Caption': 'A person blows their horn while in traffic, peoples voices are very faint in the background'}
980 {'Dataset': 'AudioCaps', 'Uid': '-QG_csJZRpw_30000_40000', 'Caption': 'A man speaks as traffic passes by'}
1297 {'Dataset': 'AudioCaps', 'Uid': '-Zx0UBTa4Mc_15000_25000', 'Caption': 'A car idling while traffic is passing by in the background'}
1341 {'Dataset': 'AudioCaps', 'Uid': '-a9HMUYHpSo_70000_80000', 

In [25]:
uid = "-8aWuKAYgfM_170000_180000"
env_wav_path = os.path.join(audiocaps_wav_path, uid + ".wav")
env_wav, sr = librosa.load(env_wav_path, sr=16000)
Audio(env_wav, rate=16000)

In [ ]:
speech_wav = recovered_audio.squeeze().cpu().numpy()
env_wav = env_wav[-len(speech_wav):]
Audio(env_wav, rate=16000)

In [27]:
mix_wav = gen_env_speech(speech_wav, env_wav, 15, 5)
sf.write("examples/recon/4_mix.wav", mix_wav, 16000)
Audio(mix_wav, rate=16000)